# 🌫️ VARG Weather / Fog Severity Classification — Local Run

This notebook is the **local version** of the Colab notebook.

### What this notebook does

1. Uses the same 5-class VARG setup and 16-frame TSM + ResNet-50 architecture.
2. Uses the **already-trained Variant 1 checkpoint** for the existing best-result evaluation with TTA.
3. Trains the **one experiment that was not completed in Colab**:
   **Variant 1 + CLAHE — Balanced Sampler + unweighted Cross-Entropy + gradient clipping**.
4. Evaluates the CLAHE model on the untouched 1,408-clip test set.
5. Runs TTA on the already-trained Variant 1 checkpoint.

The original notebook used 4,445 cleaned training clips and 1,408 cleaned test clips, with a 15% stratified validation split from the training set. The five classes are `Clear`, `Rain Moderate`, `Rain Heavy`, `Fog Moderate`, and `Fog Heavy`. 


## 1. Install / import dependencies

Run the installation cell once if these packages are not already installed.

**Windows note:** this notebook intentionally uses `num_workers=0` by default. This is slower than multiprocessing but avoids common Windows/Jupyter DataLoader issues.


In [ ]:
# If needed, run once in your local environment:
# %pip install torch torchvision opencv-python pandas numpy scikit-learn matplotlib seaborn tqdm

import os
import time
import random
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.models import resnet50

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("OpenCV :", cv2.__version__)


## 2. Local paths and configuration

### Change only these paths

Set `PROJECT_DIR` to your local `FOG_CLASSIFIER` folder.

Expected structure:

```text
FOG_CLASSIFIER/
├── VARG_Dataset/
│   ├── data_split/
│   │   └── multi_label/
│   │       ├── multi_label_train.csv
│   │       └── multi_label_test.csv
│   └── videos/
│       ├── <video>.mp4
│       └── ...
├── TSM_ResNet50_balanced_best.pth
└── (CLAHE checkpoint will be created here)
```

If your folders are arranged differently, change `TRAIN_CSV`, `TEST_CSV`, and `VIDEO_DIR` directly.


In [ ]:
# ============================================================
# LOCAL CONFIGURATION
# ============================================================

# CHANGE THIS:
PROJECT_DIR = Path(r"C:\YOUR\PATH\TO\FOG_CLASSIFIER")

TRAIN_CSV = PROJECT_DIR / "VARG_Dataset" / "data_split" / "multi_label" / "multi_label_train.csv"
TEST_CSV  = PROJECT_DIR / "VARG_Dataset" / "data_split" / "multi_label" / "multi_label_test.csv"
VIDEO_DIR = PROJECT_DIR / "VARG_Dataset" / "videos"

# Already-trained Variant 1 checkpoint from Colab
VARIANT1_CHECKPOINT = PROJECT_DIR / "TSM_ResNet50_balanced_best.pth"

# New checkpoint produced by this local CLAHE experiment
CLAHE_CHECKPOINT = PROJECT_DIR / "variant1_clahe_best_local.pth"

NUM_FRAMES = 16
IMAGE_SIZE = 224
BATCH_SIZE = 4
NUM_WORKERS = 0

# Training settings copied from the original CLAHE experiment
CLAHE_EPOCHS = 5
CLAHE_LR = 1e-4
CLAHE_WEIGHT_DECAY = 0.0
GRAD_CLIP = 5.0

CLASS_NAMES = [
    "Clear",
    "Rain Moderate",
    "Rain Heavy",
    "Fog Moderate",
    "Fog Heavy",
]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

print("Project:", PROJECT_DIR)
print("Train CSV:", TRAIN_CSV)
print("Test CSV :", TEST_CSV)
print("Video dir:", VIDEO_DIR)
print("Variant 1 checkpoint:", VARIANT1_CHECKPOINT)
print("CLAHE checkpoint:", CLAHE_CHECKPOINT)


In [ ]:
# ============================================================
# CHECK LOCAL PATHS BEFORE DOING ANYTHING EXPENSIVE
# ============================================================

for name, path in {
    "PROJECT_DIR": PROJECT_DIR,
    "TRAIN_CSV": TRAIN_CSV,
    "TEST_CSV": TEST_CSV,
    "VIDEO_DIR": VIDEO_DIR,
    "VARIANT1_CHECKPOINT": VARIANT1_CHECKPOINT,
}.items():
    print(f"{name:24s}: {'OK' if path.exists() else 'MISSING'} -> {path}")

assert TRAIN_CSV.exists(), f"Missing TRAIN_CSV: {TRAIN_CSV}"
assert TEST_CSV.exists(), f"Missing TEST_CSV: {TEST_CSV}"
assert VIDEO_DIR.exists(), f"Missing VIDEO_DIR: {VIDEO_DIR}"
assert VARIANT1_CHECKPOINT.exists(), (
    f"Missing existing Variant 1 checkpoint: {VARIANT1_CHECKPOINT}"
)

print("\n✅ Required local files/folders found.")


## 3. Device

The code automatically uses CUDA if a local NVIDIA GPU is available; otherwise it uses CPU.

Because the model/checkpoint is already trained, **loading the checkpoint does not require retraining the previous variants**.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
else:
    print("⚠️ CUDA not available — local CLAHE training will run on CPU.")


## 4. Load and clean VARG labels

This is the same cleaning logic used in the Colab notebook: only rows with **exactly one active weather label** are retained. Zero-label and multi-label rows are removed.


In [ ]:
def prepare_labels(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    active = df[CLASS_NAMES].fillna(0).astype(int).sum(axis=1)

    clean = df[active == 1].copy()
    clean["class_name"] = clean[CLASS_NAMES].idxmax(axis=1)
    clean["label"] = clean["class_name"].map(CLASS_TO_ID).astype(int)

    clean = clean[
        ["filename", "num_frames", "class_name", "label"]
    ].reset_index(drop=True)

    removed = int((active != 1).sum())

    return df, clean, removed


train_raw, train_df, train_removed = prepare_labels(TRAIN_CSV)
test_raw, test_df, test_removed = prepare_labels(TEST_CSV)

print("Original train:", len(train_raw))
print("Clean train   :", len(train_df))
print("Removed train :", train_removed)

print("\nOriginal test :", len(test_raw))
print("Clean test    :", len(test_df))
print("Removed test  :", test_removed)

print("\nTRAIN distribution:")
print(train_df["class_name"].value_counts().reindex(CLASS_NAMES))

print("\nTEST distribution:")
print(test_df["class_name"].value_counts().reindex(CLASS_NAMES))

assert len(train_df) == 4445, f"Unexpected clean train size: {len(train_df)}"
assert len(test_df) == 1408, f"Unexpected clean test size: {len(test_df)}"


## 5. Verify that all labelled videos exist


In [ ]:
video_stems = {p.stem for p in VIDEO_DIR.glob("*.mp4")}

train_exists = train_df["filename"].isin(video_stems)
test_exists = test_df["filename"].isin(video_stems)

print("MP4 files found:", len(video_stems))
print("Train missing :", int((~train_exists).sum()))
print("Test missing  :", int((~test_exists).sum()))

assert train_exists.all(), "Some cleaned training samples have no MP4."
assert test_exists.all(), "Some cleaned test samples have no MP4."

print("✅ Every cleaned train/test sample has an MP4.")


## 6. Recreate the exact train / validation / untouched test split

The original notebook used:

- 85% of cleaned train → training = 3,778 clips
- 15% of cleaned train → validation = 667 clips
- cleaned test remains untouched = 1,408 clips
- `random_state=42`
- stratification by class


In [ ]:
train_part, val_part = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df["label"],
)

train_part = train_part.reset_index(drop=True)
val_part = val_part.reset_index(drop=True)

print("Train      :", len(train_part))
print("Validation :", len(val_part))
print("Final test :", len(test_df))

assert len(train_part) == 3778
assert len(val_part) == 667
assert len(test_df) == 1408


## 7. Base MP4 video dataset

Each clip is decoded directly from MP4 and exactly 16 frames are sampled.

The normal dataset uses ImageNet normalization and the same temporal sampling scheme as the Colab notebook.


In [ ]:
IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406]
).view(3, 1, 1, 1)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225]
).view(3, 1, 1, 1)


class VARGVideoDataset(Dataset):
    def __init__(
        self,
        df,
        video_dir,
        num_frames=16,
        train=False,
        size=224,
    ):
        self.df = df.reset_index(drop=True)
        self.video_dir = Path(video_dir)
        self.num_frames = num_frames
        self.train = train
        self.size = size

    def __len__(self):
        return len(self.df)

    def _sample_indices(self, n):
        if n <= 0:
            raise RuntimeError("Video has no readable frames")

        if n >= self.num_frames:
            if self.train:
                indices = np.sort(
                    np.random.choice(
                        n,
                        self.num_frames,
                        replace=False,
                    )
                )
            else:
                indices = np.linspace(
                    0,
                    n - 1,
                    self.num_frames,
                ).astype(int)
        else:
            indices = np.array(
                list(range(n))
                + [n - 1] * (self.num_frames - n)
            )

        return indices

    def _read_frames(self, video_path):
        cap = cv2.VideoCapture(str(video_path))

        if not cap.isOpened():
            raise RuntimeError(f"Cannot open video: {video_path}")

        all_frames = []

        while True:
            ok, frame = cap.read()

            if not ok:
                break

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB,
            )

            frame = cv2.resize(
                frame,
                (self.size, self.size),
                interpolation=cv2.INTER_AREA,
            )

            all_frames.append(frame)

        cap.release()

        if not all_frames:
            raise RuntimeError(
                f"No readable frames in: {video_path}"
            )

        return all_frames

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        video_path = (
            self.video_dir / f"{row.filename}.mp4"
        )

        all_frames = self._read_frames(video_path)

        indices = self._sample_indices(len(all_frames))

        frames = [all_frames[i] for i in indices]

        # Same whole-clip horizontal flip used by the original dataset
        if self.train and random.random() < 0.5:
            frames = [
                np.ascontiguousarray(frame[:, ::-1])
                for frame in frames
            ]

        x = torch.from_numpy(
            np.stack(frames)
        ).permute(
            3, 0, 1, 2
        ).float() / 255.0

        x = (x - IMAGENET_MEAN) / IMAGENET_STD

        y = torch.tensor(
            int(row.label),
            dtype=torch.long,
        )

        return x, y


## 8. CLAHE dataset

This reproduces the uncompleted experiment from the Colab notebook.

CLAHE is applied to **train, validation, and test** frames consistently, using:

- LAB colour space
- `clipLimit=2.0`
- `tileGridSize=(8, 8)`

The balanced sampler is used only for training. Validation and test remain non-sampled.


In [ ]:
def apply_clahe(frame):
    # frame: RGB uint8, H x W x 3
    lab = cv2.cvtColor(frame, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8),
    )

    l2 = clahe.apply(l)
    lab2 = cv2.merge((l2, a, b))

    return cv2.cvtColor(
        lab2,
        cv2.COLOR_LAB2RGB,
    )


class VARGVideoDatasetCLAHE(VARGVideoDataset):
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        video_path = (
            self.video_dir / f"{row.filename}.mp4"
        )

        cap = cv2.VideoCapture(str(video_path))

        if not cap.isOpened():
            raise RuntimeError(
                f"Cannot open video: {video_path}"
            )

        all_frames = []

        while True:
            ok, frame = cap.read()

            if not ok:
                break

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB,
            )

            frame = cv2.resize(
                frame,
                (self.size, self.size),
                interpolation=cv2.INTER_AREA,
            )

            # CLAHE is applied consistently to every split.
            frame = apply_clahe(frame)

            all_frames.append(frame)

        cap.release()

        if not all_frames:
            raise RuntimeError(
                f"No readable frames in: {video_path}"
            )

        indices = self._sample_indices(len(all_frames))
        frames = [all_frames[i] for i in indices]

        if self.train and random.random() < 0.5:
            frames = [
                np.ascontiguousarray(frame[:, ::-1])
                for frame in frames
            ]

        x = torch.from_numpy(
            np.stack(frames)
        ).permute(
            3, 0, 1, 2
        ).float() / 255.0

        x = (x - IMAGENET_MEAN) / IMAGENET_STD

        y = torch.tensor(
            int(row.label),
            dtype=torch.long,
        )

        return x, y


## 9. Build CLAHE datasets and DataLoaders


In [ ]:
train_ds_clahe = VARGVideoDatasetCLAHE(
    train_part,
    VIDEO_DIR,
    num_frames=NUM_FRAMES,
    train=True,
    size=IMAGE_SIZE,
)

val_ds_clahe = VARGVideoDatasetCLAHE(
    val_part,
    VIDEO_DIR,
    num_frames=NUM_FRAMES,
    train=False,
    size=IMAGE_SIZE,
)

test_ds_clahe = VARGVideoDatasetCLAHE(
    test_df,
    VIDEO_DIR,
    num_frames=NUM_FRAMES,
    train=False,
    size=IMAGE_SIZE,
)

# Balanced sampler — training only
class_counts = (
    train_part["label"]
    .value_counts()
    .reindex(range(5), fill_value=0)
)

sampler_class_weights = 1.0 / class_counts.values

sample_weights = train_part["label"].map(
    dict(enumerate(sampler_class_weights))
).values

sample_weights = torch.DoubleTensor(sample_weights)

sampler_clahe = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader_clahe = DataLoader(
    train_ds_clahe,
    batch_size=BATCH_SIZE,
    sampler=sampler_clahe,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader_clahe = DataLoader(
    val_ds_clahe,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader_clahe = DataLoader(
    test_ds_clahe,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Class counts:")
print(class_counts)

print("\nSampling weights:")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:15s}: {sampler_class_weights[i]:.6f}")

print("\nLoaders ready.")


## 10. Exact TSM + ResNet-50 architecture

This is the same architecture used in the Colab notebook:

- ImageNet ResNet-50 backbone
- TSM inserted into every ResNet stage
- 16 temporal segments
- temporal mean consensus
- 5-class classifier
- dropout = 0.5

`weights=None` is intentional here: the saved checkpoint already contains the trained backbone weights, so local execution does not need to download ImageNet weights merely to overwrite them.


In [ ]:
class TemporalShift(nn.Module):
    def __init__(
        self,
        channels,
        num_segments=16,
        shift_div=8,
    ):
        super().__init__()

        self.num_segments = num_segments
        self.shift_div = shift_div

    def forward(self, x):
        BT, C, H, W = x.shape

        if BT % self.num_segments != 0:
            raise ValueError(
                f"{BT} samples cannot be grouped into "
                f"{self.num_segments} temporal segments"
            )

        B = BT // self.num_segments

        x = x.view(
            B,
            self.num_segments,
            C,
            H,
            W,
        )

        fold = C // self.shift_div

        out = torch.zeros_like(x)

        # Earlier -> later
        out[:, 1:, :fold] = x[:, :-1, :fold]

        # Later -> earlier
        out[:, :-1, fold:2 * fold] = x[:, 1:, fold:2 * fold]

        # No shift
        out[:, :, 2 * fold:] = x[:, :, 2 * fold:]

        return out.reshape(
            BT,
            C,
            H,
            W,
        )


class TSMResNet50(nn.Module):
    def __init__(
        self,
        num_classes=5,
        num_segments=16,
    ):
        super().__init__()

        self.num_segments = num_segments

        # No download required. Checkpoint weights replace these weights.
        self.backbone = resnet50(weights=None)

        for layer in [
            self.backbone.layer1,
            self.backbone.layer2,
            self.backbone.layer3,
            self.backbone.layer4,
        ]:
            for block in layer:
                block.conv1 = nn.Sequential(
                    TemporalShift(
                        block.conv1.in_channels,
                        num_segments=num_segments,
                    ),
                    block.conv1,
                )

        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(
                in_features,
                num_classes,
            ),
        )

    def forward(self, x):
        # x = B,C,T,H,W
        B, C, T, H, W = x.shape

        if T != self.num_segments:
            raise ValueError(
                f"Expected {self.num_segments} frames, got {T}"
            )

        x = x.permute(
            0, 2, 1, 3, 4
        )

        x = x.reshape(
            B * T,
            C,
            H,
            W,
        )

        features = self.backbone(x)

        features = features.view(
            B,
            T,
            -1,
        )

        features = features.mean(dim=1)

        return self.classifier(features)


## 11. Sanity-check the CLAHE pipeline

This only reads **one batch**. It does not train anything.


In [ ]:
videos, labels = next(iter(train_loader_clahe))

print("Input :", videos.shape)
print("Labels:", labels.shape)

with torch.no_grad():
    test_model = TSMResNet50(
        num_classes=5,
        num_segments=NUM_FRAMES,
    ).to(device)

    outputs = test_model(videos.to(device))

print("Output:", outputs.shape)

assert videos.shape == (
    min(BATCH_SIZE, len(train_ds_clahe)),
    3,
    NUM_FRAMES,
    IMAGE_SIZE,
    IMAGE_SIZE,
)

assert outputs.shape[1] == 5

del test_model, outputs, videos, labels
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ CLAHE dataset + model forward pass successful.")


# PART A — VARIANT 1 + CLAHE

## 12. Train the previously uncompleted CLAHE experiment

Recipe from the original notebook:

- Balanced `WeightedRandomSampler`
- **Unweighted** `CrossEntropyLoss`
- AdamW
- learning rate = `1e-4`
- `ReduceLROnPlateau`
- gradient clipping with `max_norm=5.0`
- 5 epochs
- best checkpoint selected by validation macro-F1


In [ ]:
model_clahe = TSMResNet50(
    num_classes=5,
    num_segments=NUM_FRAMES,
).to(device)

criterion_clahe = nn.CrossEntropyLoss()

optimizer_clahe = torch.optim.AdamW(
    model_clahe.parameters(),
    lr=CLAHE_LR,
    weight_decay=CLAHE_WEIGHT_DECAY,
)

scheduler_clahe = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_clahe,
    mode="max",
    factor=0.5,
    patience=1,
)

def evaluate_clahe(loader):
    model_clahe.eval()

    losses = []
    preds = []
    targets = []

    with torch.no_grad():
        for videos, labels in tqdm(
            loader,
            desc="Evaluation",
            leave=False,
        ):
            videos = videos.to(
                device,
                non_blocking=True,
            )
            labels = labels.to(
                device,
                non_blocking=True,
            )

            logits = model_clahe(videos)

            loss = criterion_clahe(
                logits,
                labels,
            )

            losses.append(loss.item())

            preds.extend(
                logits.argmax(1).cpu().numpy()
            )

            targets.extend(
                labels.cpu().numpy()
            )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            targets,
            preds,
            labels=list(range(5)),
            average="macro",
            zero_division=0,
        )
    )

    return (
        np.mean(losses),
        accuracy_score(targets, preds),
        precision,
        recall,
        f1,
    )


history_clahe = []
best_f1 = -1.0

print("Starting CLAHE training...")
print("Checkpoint:", CLAHE_CHECKPOINT)

for epoch in range(1, CLAHE_EPOCHS + 1):
    model_clahe.train()

    running_loss = 0.0
    n_samples = 0

    start = time.time()

    for videos, labels in tqdm(
        train_loader_clahe,
        desc=f"[CLAHE] Epoch {epoch}/{CLAHE_EPOCHS}",
    ):
        videos = videos.to(
            device,
            non_blocking=True,
        )
        labels = labels.to(
            device,
            non_blocking=True,
        )

        optimizer_clahe.zero_grad(
            set_to_none=True
        )

        logits = model_clahe(videos)

        loss = criterion_clahe(
            logits,
            labels,
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model_clahe.parameters(),
            max_norm=GRAD_CLIP,
        )

        optimizer_clahe.step()

        running_loss += (
            loss.item() * labels.size(0)
        )

        n_samples += labels.size(0)

    train_loss = running_loss / n_samples

    (
        val_loss,
        val_acc,
        val_precision,
        val_recall,
        val_f1,
    ) = evaluate_clahe(val_loader_clahe)

    scheduler_clahe.step(val_f1)

    history_clahe.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_macro_precision": val_precision,
        "val_macro_recall": val_recall,
        "val_macro_f1": val_f1,
        "lr": optimizer_clahe.param_groups[0]["lr"],
    })

    if val_f1 > best_f1:
        best_f1 = val_f1

        torch.save(
            {
                "model_state_dict": model_clahe.state_dict(),
                "class_names": CLASS_NAMES,
                "num_segments": NUM_FRAMES,
                "val_f1": best_f1,
            },
            CLAHE_CHECKPOINT,
        )

        print(
            f"💾 Best CLAHE checkpoint saved "
            f"(val_f1={best_f1:.4f}) -> {CLAHE_CHECKPOINT}"
        )

    elapsed = time.time() - start

    print(
        f"[CLAHE] Epoch {epoch}: "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.4f} | "
        f"macro_F1={val_f1:.4f} | "
        f"time={elapsed/60:.1f} min"
    )

history_clahe_df = pd.DataFrame(history_clahe)
history_clahe_df


## 13. Plot CLAHE training history


In [ ]:
if len(history_clahe_df) > 0:
    plt.figure(figsize=(7, 5))
    plt.plot(
        history_clahe_df["epoch"],
        history_clahe_df["train_loss"],
        marker="o",
        label="Train Loss",
    )
    plt.plot(
        history_clahe_df["epoch"],
        history_clahe_df["val_loss"],
        marker="o",
        label="Validation Loss",
    )
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Variant 1 + CLAHE — Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.plot(
        history_clahe_df["epoch"],
        history_clahe_df["val_macro_f1"],
        marker="o",
        label="Validation Macro F1",
    )
    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")
    plt.title("Variant 1 + CLAHE — Validation Macro F1")
    plt.legend()
    plt.grid(True)
    plt.show()


## 14. Evaluate the best CLAHE checkpoint on the untouched TEST set


In [ ]:
assert CLAHE_CHECKPOINT.exists(), (
    "CLAHE checkpoint does not exist. Run the training cell first."
)

checkpoint = torch.load(
    CLAHE_CHECKPOINT,
    map_location=device,
    weights_only=False,
)

print(
    "Loaded CLAHE checkpoint val_f1:",
    checkpoint["val_f1"],
)

model_clahe = TSMResNet50(
    num_classes=5,
    num_segments=NUM_FRAMES,
).to(device)

model_clahe.load_state_dict(
    checkpoint["model_state_dict"]
)

model_clahe.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for videos, labels in tqdm(
        test_loader_clahe,
        desc="Evaluating CLAHE model on TEST",
    ):
        videos = videos.to(
            device,
            non_blocking=True,
        )

        logits = model_clahe(videos)

        preds = (
            logits.argmax(1)
            .cpu()
            .numpy()
        )

        all_preds.extend(preds)
        all_targets.extend(
            labels.numpy()
        )

clahe_test_accuracy = accuracy_score(
    all_targets,
    all_preds,
)

clahe_balanced_accuracy = balanced_accuracy_score(
    all_targets,
    all_preds,
)

clahe_precision, clahe_recall, clahe_f1, _ = (
    precision_recall_fscore_support(
        all_targets,
        all_preds,
        labels=list(range(5)),
        average="macro",
        zero_division=0,
    )
)

print("\n========== VARIANT 1 + CLAHE ==========")
print(f"Test Accuracy     : {clahe_test_accuracy:.4f}")
print(f"Balanced Accuracy : {clahe_balanced_accuracy:.4f}")
print(f"Macro Precision   : {clahe_precision:.4f}")
print(f"Macro Recall      : {clahe_recall:.4f}")
print(f"Macro F1          : {clahe_f1:.4f}")

print("\nPer-class report:")
print(
    classification_report(
        all_targets,
        all_preds,
        target_names=CLASS_NAMES,
        digits=3,
        zero_division=0,
    )
)

cm_clahe = confusion_matrix(
    all_targets,
    all_preds,
    labels=list(range(5)),
)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_clahe,
    annot=True,
    fmt="d",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Variant 1 + CLAHE — TEST Confusion Matrix")
plt.tight_layout()
plt.show()


# PART B — EXISTING BEST RESULT: VARIANT 1 + TTA

The Colab notebook's recorded TTA result was:

- **Test Accuracy: 0.6825**
- **Macro F1: 0.6532**

TTA itself does **not retrain the model**. It re-samples each test clip using multiple temporal offsets, optionally includes a horizontal flip, averages the softmax probabilities, and then selects the class with the highest averaged probability. 


In [ ]:
# ============================================================
# Load the ALREADY-TRAINED Variant 1 checkpoint
# ============================================================

variant1_model = TSMResNet50(
    num_classes=5,
    num_segments=NUM_FRAMES,
).to(device)

variant1_checkpoint = torch.load(
    VARIANT1_CHECKPOINT,
    map_location=device,
    weights_only=False,
)

print(
    "Checkpoint stored validation F1:",
    variant1_checkpoint.get("val_f1", "not stored"),
)

variant1_model.load_state_dict(
    variant1_checkpoint["model_state_dict"]
)

variant1_model.eval()

print("✅ Existing Variant 1 checkpoint loaded.")


In [ ]:
@torch.no_grad()
def evaluate_with_tta(
    df,
    video_dir,
    model,
    device,
    num_frames=16,
    size=224,
    n_offsets=2,
    use_flip=True,
):
    model.eval()

    preds = []
    targets = []

    for idx in tqdm(
        range(len(df)),
        desc="TTA evaluation",
    ):
        row = df.iloc[idx]

        video_path = (
            Path(video_dir)
            / f"{row.filename}.mp4"
        )

        cap = cv2.VideoCapture(
            str(video_path)
        )

        frames = []

        while True:
            ok, frame = cap.read()

            if not ok:
                break

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB,
            )

            frame = cv2.resize(
                frame,
                (size, size),
                interpolation=cv2.INTER_AREA,
            )

            frames.append(frame)

        cap.release()

        n = len(frames)

        if n == 0:
            raise RuntimeError(
                f"No readable frames in {video_path}"
            )

        base = np.linspace(
            0,
            max(n - 1, 0),
            num_frames,
        ).astype(int)

        step = max(
            1,
            n // (num_frames * 2),
        )

        offsets = [
            0,
            step,
            -step,
        ][:max(1, n_offsets)]

        probs_accum = torch.zeros(
            len(CLASS_NAMES)
        )

        views = 0

        for off in offsets:
            idxs = np.clip(
                base + off,
                0,
                n - 1,
            )

            clip = [
                frames[i]
                for i in idxs
            ]

            variants = [clip]

            if use_flip:
                variants.append(
                    [
                        np.ascontiguousarray(
                            f[:, ::-1]
                        )
                        for f in clip
                    ]
                )

            for v in variants:
                x = torch.from_numpy(
                    np.stack(v)
                ).permute(
                    3, 0, 1, 2
                ).float() / 255.0

                x = (
                    x - IMAGENET_MEAN
                ) / IMAGENET_STD

                x = x.unsqueeze(0).to(
                    device
                )

                logits = model(x)

                probs_accum += (
                    torch.softmax(
                        logits,
                        dim=1,
                    )[0].cpu()
                )

                views += 1

        preds.append(
            int(
                (probs_accum / views)
                .argmax()
            )
        )

        targets.append(
            int(row.label)
        )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            targets,
            preds,
            labels=list(range(5)),
            average="macro",
            zero_division=0,
        )
    )

    accuracy = accuracy_score(
        targets,
        preds,
    )

    balanced_accuracy = balanced_accuracy_score(
        targets,
        preds,
    )

    return (
        accuracy,
        balanced_accuracy,
        precision,
        recall,
        f1,
        preds,
        targets,
    )


## 15. Run Variant 1 + TTA

This is the expensive part of evaluation: the 1,408 test clips are each evaluated using multiple temporal views and horizontal-flip views.

The settings below reproduce the Colab TTA recipe:

- `n_offsets=2`
- `use_flip=True`


In [ ]:
(
    tta_accuracy,
    tta_balanced_accuracy,
    tta_precision,
    tta_recall,
    tta_f1,
    tta_preds,
    tta_targets,
) = evaluate_with_tta(
    test_df,
    VIDEO_DIR,
    variant1_model,
    device,
    num_frames=NUM_FRAMES,
    size=IMAGE_SIZE,
    n_offsets=2,
    use_flip=True,
)

print("\n========== VARIANT 1 + TTA ==========")
print(f"Test Accuracy     : {tta_accuracy:.4f}")
print(f"Balanced Accuracy : {tta_balanced_accuracy:.4f}")
print(f"Macro Precision   : {tta_precision:.4f}")
print(f"Macro Recall      : {tta_recall:.4f}")
print(f"Macro F1          : {tta_f1:.4f}")

print("\nPer-class report:")
print(
    classification_report(
        tta_targets,
        tta_preds,
        target_names=CLASS_NAMES,
        digits=3,
        zero_division=0,
    )
)

cm_tta = confusion_matrix(
    tta_targets,
    tta_preds,
    labels=list(range(5)),
)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_tta,
    annot=True,
    fmt="d",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Variant 1 + TTA — TEST Confusion Matrix")
plt.tight_layout()
plt.show()


## 16. Final comparison

This compares the **newly trained CLAHE experiment** with the **existing Variant 1 + TTA result**.

The original Colab notebook recorded Variant 1 + TTA as `0.6825` test accuracy and `0.6532` macro F1. Your local run should be treated as the current measured result for the exact local files/checkpoint you use.


In [ ]:
comparison = pd.DataFrame([
    {
        "Experiment": "Variant 1 + CLAHE",
        "Test Accuracy": clahe_test_accuracy,
        "Balanced Accuracy": clahe_balanced_accuracy,
        "Macro Precision": clahe_precision,
        "Macro Recall": clahe_recall,
        "Macro F1": clahe_f1,
    },
    {
        "Experiment": "Variant 1 + TTA",
        "Test Accuracy": tta_accuracy,
        "Balanced Accuracy": tta_balanced_accuracy,
        "Macro Precision": tta_precision,
        "Macro Recall": tta_recall,
        "Macro F1": tta_f1,
    },
])

comparison


## Notes

- **Do not retrain the previous Variant 1 model.** Its checkpoint is loaded from `TSM_ResNet50_balanced_best.pth`.
- **Do not use the CLAHE checkpoint for TTA unless you specifically want CLAHE + TTA.** The TTA section intentionally evaluates the already-trained Variant 1 checkpoint because that is the model associated with the existing best TTA result.
- The untouched test set contains 1,408 cleaned clips.
- The CLAHE experiment is the only training experiment in this local notebook.
- If training on CPU is too slow, reduce `CLAHE_EPOCHS` for a smoke test first; do not use the smoke-test result as your final experiment.
